# 3.0 — Walk-Forward Backtest Results

Run the walk-forward engine on processed features, compare against
factor benchmarks, and produce diagnostic plots.

Loads processed data from `data/processed/` and interim prices from
`data/interim/`. Can also load pre-computed model outputs from `models/`
if the backtest has already been run via `make all`.

In [ ]:
%load_ext autoreload
%autoreload 2
%cd ..

In [ ]:
import pickle
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from stock_data.config import N_BOOT
from stock_data.modeling.train import walk_forward, factor_benchmarks
from stock_data.evaluation import summarize_walk_forward, evaluate_factors
from stock_data.plots import plot_walk_forward_diagnostics

## Load Data

In [ ]:
PROCESSED = Path("data/processed")
INTERIM = Path("data/interim")
MODELS = Path("models")

risk_model_df = pd.read_parquet(PROCESSED / "risk_model_df.parquet")
with open(PROCESSED / "feature_cols.pkl", "rb") as f:
    feature_cols_all = pickle.load(f)
close_prices = pd.read_parquet(INTERIM / "close_prices.parquet")

print(f"Loaded: {risk_model_df.shape[0]} rows, {len(feature_cols_all)} features")

## Walk-Forward Engine

In [ ]:
prod_df, prod_fi, prod_weights_history = walk_forward(
    risk_model_df, feature_cols_all, close_prices,
)
summarize_walk_forward(prod_df, prod_fi, feature_cols_all)

## Factor Benchmarks & Bootstrap

In [ ]:
factor_results = factor_benchmarks(risk_model_df, feature_cols_all, prod_df)
ci_lo, ci_hi, boot_means, ex_n = evaluate_factors(prod_df, factor_results)

## Diagnostic Plots

In [ ]:
fig = plot_walk_forward_diagnostics(
    prod_df, factor_results, boot_means, ci_lo, ci_hi, ex_n,
)
fig.savefig("reports/figures/walk_forward_diagnostics.png", dpi=150, bbox_inches="tight")
plt.show()

## Save Model Outputs

In [ ]:
MODELS.mkdir(parents=True, exist_ok=True)

prod_df.to_parquet(MODELS / "prod_df.parquet")
with open(MODELS / "prod_fi.pkl", "wb") as f:
    pickle.dump(prod_fi, f)
with open(MODELS / "weights_history.pkl", "wb") as f:
    pickle.dump(prod_weights_history, f)
with open(MODELS / "factor_results.pkl", "wb") as f:
    pickle.dump(factor_results, f)

print(f"Model outputs saved to {MODELS}/")